# Local Track

Goal:

- Retrain the YOLO model
- Track the training with mlflow


## Environment

Setup notebook and mlflow with `docker compose`

> Jupyter at http://127.0.0.1:8888. `MLFLOW_TRACKING_URI` is set by compose.


In [ ]:
from src.tracking import tracking_uri
import sys
from pathlib import Path

import mlflow
import torch
import ultralytics

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

# print version
print("python     ", sys.version.split()[0])
print("torch      ", torch.__version__)
print("ultralytics", ultralytics.__version__)
print("mlflow     ", mlflow.__version__)
print("cuda       ", torch.cuda.is_available())

# print mlflow uri
print("tracking   ", tracking_uri())

WARNING ⚠️ user config directory '/workspace/.ultralytics/Ultralytics' is not writable, using '/tmp/Ultralytics'. Set YOLO_CONFIG_DIR to override.
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/tmp/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
python      3.12.13
torch       2.13.0+cu130
ultralytics 8.4.116
mlflow      3.15.1
cuda        False
tracking    http://mlflow:5000


In [ ]:
# define MLflow instance
mlflow.set_tracking_uri(tracking_uri())

# test by search experiments
print(mlflow.search_experiments())

[<Experiment: artifact_location='/mlflow/artifacts/4', creation_time=1786166268190, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1786166268190, lifecycle_stage='active', name='yolo-plate-tuning', tags={}, trace_location=None, workspace='default'>, <Experiment: artifact_location='/mlflow/artifacts/2', creation_time=1786163766119, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1786163766119, lifecycle_stage='active', name='yolo-plate-detection', tags={}, trace_location=None, workspace='default'>, <Experiment: artifact_location='/mlflow/artifacts/0', creation_time=1786162182159, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1786162182159, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]


## Data summary

Inspect the raw dataset before training.


In [ ]:
from src.data_loader import build_split, summarize, verify_split, write_data_yaml

LIMIT = None

# print summary
stats = summarize(RAW)
print({k: stats[k] for k in ("pairs", "boxes_total", "boxes_per_image_max", "malformed")})

# print split
print(build_split(RAW, PROCESSED, val_fraction=0.2, limit=LIMIT, seed=0))
print(verify_split(PROCESSED))

# print data config
names = (RAW / "classes.txt").read_text().split()
data_yaml = write_data_yaml(ROOT / "configs" / "data.yaml", PROCESSED, names)
print(data_yaml.read_text())

{'pairs': 556, 'boxes_total': 574, 'boxes_per_image_max': 3, 'malformed': []}
{'train': 445, 'val': 111, 'orphan_images': 0, 'orphan_labels': 0}
{'train': 445, 'val': 111}
path: /workspace/data/processed
train: train/images
val: val/images
nc: 1
names: ['car_plate']



## Train with tracking

Use mlflow callback to pass training process


In [ ]:
import os
import time

import yaml
from ultralytics import YOLO

# experiment name
EXPERIMENT = "yolo-plate-detection"

# load parameters from config file
train_cfg = yaml.safe_load((ROOT / "configs" / "train.yaml").read_text())
cfg = dict(train_cfg)
model_weights = cfg.pop("model")
cfg["project"] = str(ROOT / cfg["project"])

n_train = len(list((PROCESSED / "train" / "images").iterdir()))

# Set env var
os.environ["MLFLOW_EXPERIMENT_NAME"] = EXPERIMENT
os.environ["MLFLOW_RUN"] = f"cpu-{n_train}img-{cfg['epochs']}ep-{cfg['imgsz']}px"

# Keep the run open
os.environ["MLFLOW_KEEP_RUN_ACTIVE"] = "true"

print(f"experiment {EXPERIMENT}")
print(f"run        {os.environ['MLFLOW_RUN']}")

# construct model class with parameters
model = YOLO(model_weights)

# log start time
start = time.time()

# run the training process, get performance results
results = model.train(data=str(data_yaml), **cfg)

# print elapsed time
print(f"\nelapsed: {time.time() - start:.0f}s")

experiment yolo-plate-detection
run        cpu-445img-10ep-416px
Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.13.0+cu130 CPU (Intel Core 5 120U)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/workspace/configs/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosai

### MLflow log hyperparameters

Log parameters and terminate run


In [ ]:
from src.tracking import log_dataset_context

# Log hyperparameters to mlflow with helping function
logged = log_dataset_context(
    PROCESSED,
    RAW,
    **{"data.limit": str(LIMIT), "data.seed": 0, "run.device": "cpu"},
)

# get currently active MLflow run
run = mlflow.active_run()

# print id
print(f"run_id {run.info.run_id}")

# print parameters
for key, value in logged.items():
    print(f"  {key:26} {value}")

# terminates the currently active MLflow run 
mlflow.end_run()
print("\nrun closed")

run_id 21ed80b3779640da84bb50ade6259c43
  data.train_images          445
  data.train_boxes           458
  data.val_images            111
  data.val_boxes             116
  data.total_images          556
  data.available_images      556
  data.fraction_used         1.0
  data.limit                 None
  data.seed                  0
  run.device                 cpu
🏃 View run cpu-445img-10ep-416px at: http://mlflow:5000/#/experiments/2/runs/21ed80b3779640da84bb50ade6259c43
🧪 View experiment at: http://mlflow:5000/#/experiments/2

run closed


Confirm parameters are logged.


In [ ]:
from src.tracking import latest_run_id

# get last run
run_id = latest_run_id(EXPERIMENT)
fetched = mlflow.get_run(run_id)

# print last run param
print(f"run_id  {run_id}")
print(f"status  {fetched.info.status}")

print("\nfinal metrics")
for key in sorted(fetched.data.metrics):
    if "mAP" in key or "precision" in key or "recall" in key:
        print(f"  {key:28} {fetched.data.metrics[key]:.4f}")

print("\ndataset params")
for key in sorted(k for k in fetched.data.params if k.startswith("data.")):
    print(f"  {key:28} {fetched.data.params[key]}")

print("\nartifacts")
for artifact in mlflow.artifacts.list_artifacts(run_id=run_id):
    print(f"  {artifact.path}")

run_id  21ed80b3779640da84bb50ade6259c43
status  FINISHED

final metrics
  metrics/mAP50-95B            0.7494
  metrics/mAP50B               0.9502
  metrics/precisionB           0.9616
  metrics/recallB              0.9310

dataset params
  data.available_images        556
  data.fraction_used           1.0
  data.limit                   None
  data.seed                    0
  data.total_images            556
  data.train_boxes             458
  data.train_images            445
  data.val_boxes               116
  data.val_images              111

artifacts
  BoxF1_curve.png
  BoxPR_curve.png
  BoxP_curve.png
  BoxR_curve.png
  args.yaml
  confusion_matrix.png
  confusion_matrix_normalized.png
  labels.jpg
  results.csv
  results.png
  train_batch0.jpg
  train_batch1.jpg
  train_batch2.jpg
  val_batch0_labels.jpg
  val_batch0_pred.jpg
  val_batch1_labels.jpg
  val_batch1_pred.jpg
  val_batch2_labels.jpg
  val_batch2_pred.jpg
  weights


Get historical metrics


In [ ]:
# Get historical metrics
client = mlflow.tracking.MlflowClient()  # clien interface
history = client.get_metric_history(run_id, "metrics/mAP50-95B")

print(f"{'epoch':>6} {'mAP50-95':>10}")
for point in history:
    print(f"{point.step:>6} {point.value:>10.4f}")

 epoch   mAP50-95
     0     0.0359
     1     0.4264
     2     0.5039
     3     0.6316
     4     0.6581
     5     0.6751
     6     0.7173
     7     0.7266
     8     0.7446
     9     0.7492
    10     0.7494


## List runs

Compare every run in the experiment, side by side.


In [8]:
from src.tracking import compare_runs

compare_runs(EXPERIMENT)

,mlflow.runName,epochs,imgsz,data.train_images,metrics/mAP50B,metrics/mAP50-95B,metrics/precisionB,metrics/recallB
0,cpu-445img-10ep-416px,10,416,445,0.950199,0.749393,0.961617,0.931034
1,cpu-445img-10ep-416px,10,416,445,0.950199,0.749393,0.961617,0.931034
2,cpu-445img-10ep-416px,10,416,None,NaN,NaN,NaN,NaN
3,cpu-445img-10ep-416px,10,416,445,0.950199,0.749393,0.961617,0.931034


Browse the same data at http://127.0.0.1:5000.